# Should I Trade Today?

**Run at 3:00–3:15 PM IST. Two cells. Done.**

| Step | What it does |
|------|--------------|
| Cell 1 | Set your capital |
| Cell 2 | Fetches NIFTY 50 scores → prints the buy list |

**Rule:** Buy the top 10 ranked stocks at 3:20 PM every trading day. Sell next morning at 9:25 AM.

---
Strategy gross mean return: **~+0.20%/session** over 7 years (2019–2026).


In [1]:
CAPITAL = 200_000   # Rs — change this to your actual capital today

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, yfinance as yf
from datetime import date

TOP_N    = 10
LOOKBACK = 20
TODAY    = pd.Timestamp(date.today())

NIFTY50_YF = [
    'ADANIENT.NS','ADANIPORTS.NS','APOLLOHOSP.NS','ASIANPAINT.NS','AXISBANK.NS',
    'BAJAJ-AUTO.NS','BAJFINANCE.NS','BAJAJFINSV.NS','BEL.NS','BHARTIARTL.NS',
    'CIPLA.NS','COALINDIA.NS','DRREDDY.NS','EICHERMOT.NS','ETERNAL.NS',
    'GRASIM.NS','HCLTECH.NS','HDFCBANK.NS','HDFCLIFE.NS','HINDALCO.NS',
    'HINDUNILVR.NS','ICICIBANK.NS','INDIGO.NS','INFY.NS','ITC.NS',
    'JIOFIN.NS','JSWSTEEL.NS','KOTAKBANK.NS','LT.NS','M%26M.NS',
    'MARUTI.NS','MAXHEALTH.NS','NESTLEIND.NS','NTPC.NS','ONGC.NS',
    'POWERGRID.NS','RELIANCE.NS','SBILIFE.NS','SHRIRAMFIN.NS','SBIN.NS',
    'SUNPHARMA.NS','TCS.NS','TATACONSUM.NS','TMPV.NS','TATASTEEL.NS',
    'TECHM.NS','TITAN.NS','TRENT.NS','ULTRACEMCO.NS','WIPRO.NS',
]

# ── Fetch NIFTY 50 OHLC ──────────────────────────────────────────────────────
raw   = yf.download(NIFTY50_YF, period='60d', progress=False, auto_adjust=True)

def clean(df):
    df = df.copy()
    df.columns = [c.replace('.NS','').replace('%26','&') for c in df.columns]
    return df

close = clean(raw['Close'])
open_ = clean(raw['Open'])

valid = close.notna().mean() > 0.7
close = close.loc[:, valid]
open_ = open_[close.columns]

scores = (open_ / close.shift(1) - 1).rolling(LOOKBACK, min_periods=LOOKBACK).mean()

# Last valid score per stock independently.
# Do NOT use iloc[-1] on the whole matrix — on live trading days the last row
# may be today with most stocks still NaN (open not yet populated by yfinance).
last_scores = scores.apply(lambda col: col.dropna().iloc[-1] if col.notna().any() else float('nan'))
last_scores = last_scores.dropna()

prev_close = close.apply(lambda col: col.dropna().iloc[-1] if col.notna().any() else float('nan'))

top   = last_scores.nlargest(TOP_N)
alloc = CAPITAL / TOP_N

# ── Print buy list ────────────────────────────────────────────────────────────
print(f'Today: {TODAY.date()}')
print()
print('*** TRADE TODAY ***')
print('Buy at 3:20 PM  |  Sell tomorrow at 9:25 AM')
print()
print(f'  {"#":<4} {"Stock":<13} {"20d Score":>10}  {"Prev Close":>11}  {"Qty to buy":>11}')
print(f'  {"-" * 55}')
for i, (sym, score) in enumerate(top.items(), 1):
    price = float(prev_close.get(sym, float('nan')))
    qty   = int(alloc / price) if price > 0 and price == price else 0
    print(f'  {i:<4} {sym:<13} {score:>+10.4%}  {price:>11,.2f}  {qty:>11}')
print(f'  {"-" * 55}')
print(f'  Capital: Rs {CAPITAL:,.0f}   Per stock: Rs {alloc:,.0f}  |  Stocks with valid scores: {len(last_scores)}/50')
